In [1]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

# Check available devices first
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of CUDA devices: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}: {torch.cuda.get_device_name(i)}")

# Set specific CUDA device
CUDA_DEVICE = 1  # Explicitly use device 1
device = f"cuda:{CUDA_DEVICE}" if torch.cuda.is_available() and torch.cuda.device_count() > CUDA_DEVICE else "cpu"
print(f"Using device: {device}")

# Set device before loading model
torch.cuda.set_device(CUDA_DEVICE)

TORCH_DTYPE = torch.float16
TRANSCRIPT_MODEL_ID = "models/whisper-large-v3-turbo"

# Load the model
transcript_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    TRANSCRIPT_MODEL_ID,
    torch_dtype=TORCH_DTYPE,
    low_cpu_mem_usage=True,
    use_safetensors=True,
)

# Move model to the specified device
transcript_model.to(device)
transcript_processor = AutoProcessor.from_pretrained(TRANSCRIPT_MODEL_ID)

# Create pipeline
transcript_pipe = pipeline(
    "automatic-speech-recognition",
    model=transcript_model,
    tokenizer=transcript_processor.tokenizer,
    feature_extractor=transcript_processor.feature_extractor,
    torch_dtype=TORCH_DTYPE,
    device=device,
)

def custom_transcribe(audio_data, sample_rate, time_offset, sentence_mode):
    """Transcribes the voice with timestamps"""
    audio_input = {"raw": audio_data, "sampling_rate": sample_rate}
    return_timestamps = True if sentence_mode else "word"
    output = transcript_pipe(audio_input, return_timestamps=return_timestamps)
    # return [chunk["text"] for chunk in output["chunks"]]
    return [
        [
            chunk["text"],
            -1 if sentence_mode else (chunk["timestamp"][0] + time_offset),
            # chunk["timestamp"][1],
        ]
        for chunk in output["chunks"]
    ]

# For long audio (30 minutes), implement chunking
def transcribe_long_audio(audio_data, sample_rate, chunk_seconds=300, sentence_mode=False):
    """Process long audio in chunks"""
    full_transcript = []
    chunk_samples = chunk_seconds * sample_rate
    
    # Process audio in chunks
    for i in range(0, len(audio_data), chunk_samples):
        print(f"Processing chunk {i//chunk_samples + 1}...")
        chunk = audio_data[i:i+chunk_samples]
        offset = chunk_seconds * (i//chunk_samples)
        chunk_text = custom_transcribe(chunk, sample_rate, offset, sentence_mode)
        full_transcript.extend(chunk_text)
    
    return full_transcript

CUDA available: True
Number of CUDA devices: 2
Device 0: NVIDIA GeForce RTX 4090
Device 1: NVIDIA GeForce RTX 4060
Using device: cuda:1


Device set to use cuda:1


In [2]:
from scipy.io import wavfile
import numpy as np

def process_file(file_name_no_ext):
    # Load WAV file
    sample_rate, audio_data = wavfile.read(f"{file_name_no_ext}.wav")
    
    # Convert to float and normalize if needed
    if audio_data.dtype == np.int16:
        audio_data = audio_data.astype(np.float32) / 32768.0
    elif audio_data.dtype == np.int32:
        audio_data = audio_data.astype(np.float32) / 2147483648.0
    
    # If stereo, convert to mono
    if len(audio_data.shape) > 1 and audio_data.shape[1] == 2:
        audio_data = audio_data.mean(axis=1)
    
    # Transcribe the long audio
    transcript_sentence = transcribe_long_audio(audio_data, sample_rate, 300, True)
    transcript_timestamps = [] # transcribe_long_audio(audio_data, sample_rate, 300, False)
    
    # Save the transcript
    with open(f"{file_name_no_ext}.txt", "w") as f:
        f.write(f"Transcript of {file_name_no_ext} made with Love for Viktoriia by Pavlo\n\n")
        f.write("Sentences: \n\n")
        for line in transcript_sentence:
            f.write(f"{line[0]}\n")
        f.write("Timeline: \n\n")
        for line in transcript_timestamps:
            seconds = int(line[1])
            
            hours, remainder = divmod(seconds, 3600)
            minutes, secs = divmod(remainder, 60)
            readable = f"{hours:02d}h {minutes:02d}m {secs:02d}s"
            f.write(f"{readable} : {line[0]} \n")
            transcript_timestamps
    
    print(f"Transcription of {file_name_no_ext}.wav complete!")

In [3]:
process_file("idea-real")

/home/pavlo/miniconda3/envs/speech_env/lib/python3.9/site-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.


Processing chunk 1...
Transcription of idea-real.wav complete!
